# Medical Embedding Model Comparison

Compare **all-MiniLM-L6-v2** (general purpose) and **NeuML/pubmedbert-base-embeddings** (biomedical) on 40 labeled clinical sentence pairs.

**Setup:** install `requirements.txt` from the project root first. This notebook is intended to be launched from the project root using `jupyter notebook`; if launched from `notebooks/`, it auto-adjusts the working directory.


## 1. Load the synthetic evaluation dataset

Each row is labeled: 1 means clinically equivalent, 0 means clinically different.


In [ ]:
from pathlib import Path
import os
import pandas as pd

if not Path("run_analysis.py").exists() and Path("../run_analysis.py").exists():
    os.chdir("..")

df = pd.read_csv("data/medical_sentence_pairs.csv")
print(df.groupby(["category", "same_meaning"]).size())
display(df.head(8))


## 2. Run both *real* pretrained models

First execution needs internet access to download model weights. The script generates numerical embeddings, 40 cosine similarity pairs, category evaluation, a report and PCA visualizations. It does not invent scores.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "run_analysis.py", "--batch-size", "8"], check=True)


## 3. Inspect cosine similarity results


In [ ]:
results = pd.read_csv("outputs/results.csv")
display(results[["pair_id", "category", "sentence_1", "sentence_2", "same_meaning", "general_cosine", "biomedical_cosine"]].head(12))


## 4. Compare ranking AUC by category

AUC compares a group to the opposite equivalence class, within the same model. Higher AUC means better separation; unlike raw cosine, it is useful across different embedding spaces.


In [ ]:
overall = pd.read_csv("outputs/overall_summary.csv")
by_category = pd.read_csv("outputs/category_comparison.csv")
display(overall)
display(by_category)


## 5. Inspect visualizations

PCA coordinates are fit separately for each model. The plots visualize relative positions, not exact original-space cosines.


In [ ]:
from IPython.display import Image, display
for name in ["pca_general", "pca_biomedical", "cosine_by_category", "auc_by_category"]:
    print(name)
    display(Image(filename=f"outputs/figures/{name}.png", width=760))


## 6. See the generated analysis report


In [ ]:
from IPython.display import Markdown
display(Markdown(Path("outputs/analysis_report.md").read_text(encoding="utf-8")))
